# Task 3.2 (a) — Default-Hyperparameter Baseline

This notebook establishes a **fair baseline** for algorithm comparison by evaluating each classifier with its default hyperparameters using R = 10 iterations of 5-fold stratified cross-validation. No hyperparameter tuning is performed — the inner loop is disabled (`n_inner=None`).

The purpose of this baseline is twofold:
1. To provide a reference point against which the tuned (rnCV) results will be compared.
2. To assess whether the gain from hyperparameter tuning is meaningful for each algorithm


In [1]:
import sys, os, time, pickle
import warnings
warnings.filterwarnings('ignore')

# Make the src/ package importable
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src import RepeatedNestedCV, get_estimator_configs, METRIC_NAMES

# Ensure results directory exists
os.makedirs('../results', exist_ok=True)

print('Setup complete.')

Setup complete.


## 1. Load Data

In [2]:
# ── Load dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('../data/students_dataset.csv')
X  = df.drop('num', axis=1)
y  = df['num'].values

print(f'Dataset: {X.shape[0]} samples × {X.shape[1]} features')
print(f'Class distribution: {pd.Series(y).value_counts().to_dict()}')

Dataset: 242 samples × 13 features
Class distribution: {0: 131, 1: 111}


## 2. Configure the Pipeline

We use the same `RepeatedNestedCV` class as the full pipeline, but disable the inner loop by setting `n_inner=None`. In this mode the class performs simple R × N-fold stratified CV: for each outer fold the algorithm is trained with default hyperparameters on the training portion and evaluated on the held-out test portion. Preprocessing (mode imputation + StandardScaler on continuous features only) is still applied strictly within each fold to prevent data leakage.

The seven algorithms required by the assignment are evaluated.

In [3]:
# Build the baseline rnCV instance
configs = get_estimator_configs(seed=42)
print('Algorithms under evaluation:')
for c in configs:
    print(f'  • {c.name}')

baseline_rncv = RepeatedNestedCV(
    estimator_configs = configs,
    n_outer           = 5,        # 5-fold outer CV
    n_inner           = None,     # ← disables inner loop = default-HP baseline
    n_repetitions     = 10,       # R = 10
    base_seed         = 42,       # reproducibility: each repetition uses base_seed + r
)
print(f'\nTotal evaluations: 7 algos × 10 reps × 5 outer folds = {7 * 10 * 5} = 350')

Algorithms under evaluation:
  • LogisticRegression
  • GaussianNB
  • LDA
  • RandomForest
  • LightGBM
  • XGBoost
  • CatBoost

Total evaluations: 7 algos × 10 reps × 5 outer folds = 350 = 350


## 3. Run the Baseline

In [4]:
# ── Run ───
t0 = time.time()
baseline_results = baseline_rncv.run(X, y)
elapsed = time.time() - t0

print(f'\nCompleted in {elapsed:.1f} s ({elapsed/60:.1f} min)')
print(f'Results shape: {baseline_results.shape}  (expected: 350 rows × 11 cols)')
baseline_results.head()

  Repetition 1/10 complete.
  Repetition 2/10 complete.
  Repetition 3/10 complete.
  Repetition 4/10 complete.
  Repetition 5/10 complete.
  Repetition 6/10 complete.
  Repetition 7/10 complete.
  Repetition 8/10 complete.
  Repetition 9/10 complete.
  Repetition 10/10 complete.

Completed in 24.0 s (0.4 min)
Results shape: (350, 11)  (expected: 350 rows × 11 cols)


,algorithm,repetition,outer_fold,mcc,auc,pr_auc,balanced_acc,f1,recall,specificity,precision
0,LogisticRegression,0,0,0.714525,0.916388,0.926373,0.857860,0.851064,0.869565,0.846154,0.833333
1,GaussianNB,0,0,0.713091,0.909699,0.902606,0.855351,0.844444,0.826087,0.884615,0.863636
2,LDA,0,0,0.754181,0.926421,0.939597,0.877090,0.869565,0.869565,0.884615,0.869565
3,RandomForest,0,0,0.720736,0.939799,0.938135,0.860368,0.857143,0.913043,0.807692,0.807692
4,LightGBM,0,0,0.590301,0.864548,0.878587,0.795151,0.782609,0.782609,0.807692,0.782609


## 4. Quick Summary

In [5]:
# ── Summary table: median + 95% bootstrap CI per algorithm ──────────────────
baseline_summary = baseline_rncv.summary(['mcc', 'auc', 'pr_auc', 'balanced_acc', 'f1'])
print('Baseline (default HPs) — median [95% CI]:')
display(baseline_summary)

Baseline (default HPs) — median [95% CI]:


,mcc,auc,pr_auc,balanced_acc,f1
algorithm,,,,,
LogisticRegression,"0.640 [0.590, 0.678]","0.892 [0.876, 0.906]","0.893 [0.866, 0.912]","0.815 [0.794, 0.833]","0.791 [0.776, 0.816]"
GaussianNB,"0.642 [0.607, 0.687]","0.892 [0.878, 0.907]","0.886 [0.860, 0.901]","0.817 [0.798, 0.842]","0.800 [0.773, 0.828]"
LDA,"0.667 [0.628, 0.707]","0.897 [0.881, 0.911]","0.897 [0.874, 0.911]","0.823 [0.809, 0.849]","0.809 [0.780, 0.829]"
RandomForest,"0.590 [0.577, 0.624]","0.878 [0.864, 0.901]","0.876 [0.861, 0.896]","0.794 [0.779, 0.809]","0.774 [0.756, 0.787]"
LightGBM,"0.553 [0.538, 0.590]","0.856 [0.837, 0.866]","0.860 [0.833, 0.871]","0.775 [0.766, 0.793]","0.750 [0.739, 0.773]"
XGBoost,"0.569 [0.523, 0.593]","0.859 [0.837, 0.876]","0.860 [0.844, 0.871]","0.779 [0.759, 0.796]","0.758 [0.744, 0.783]"
CatBoost,"0.588 [0.550, 0.626]","0.877 [0.865, 0.892]","0.875 [0.865, 0.906]","0.787 [0.774, 0.809]","0.769 [0.747, 0.785]"


In [6]:
# ── Quick ranked view by median MCC ──────────────────────────────────────────
print('Algorithms ranked by median MCC (default HPs):')
ranking = (baseline_results.groupby('algorithm')['mcc']
            .median().sort_values(ascending=False))
for i, (algo, mcc) in enumerate(ranking.items(), 1):
    print(f'  {i}. {algo:<22s}  median MCC = {mcc:.3f}')

Algorithms ranked by median MCC (default HPs):
  1. LDA                     median MCC = 0.667
  2. GaussianNB              median MCC = 0.642
  3. LogisticRegression      median MCC = 0.640
  4. RandomForest            median MCC = 0.590
  5. CatBoost                median MCC = 0.588
  6. XGBoost                 median MCC = 0.569
  7. LightGBM                median MCC = 0.553


## 5. Save Results

Results are saved to `../results/baseline_results.pkl` for use by `task3c_analysis.ipynb`.

In [7]:
# ── Save results ────
out_path = '../results/baseline_results.pkl'
with open(out_path, 'wb') as f:
    pickle.dump({
        'results':  baseline_results,
        'summary':  baseline_summary,
        'config': {
            'n_outer':       5,
            'n_inner':       None,
            'n_repetitions': 10,
            'base_seed':     42,
            'algorithms':    [c.name for c in configs],
        },
        'runtime_seconds': elapsed,
    }, f)
print(f'✓ Saved baseline results to {out_path}')

✓ Saved baseline results to ../results/baseline_results.pkl
